# Ejercicio propuesto

Sobre el CSV de jugadores de baloncesto que se te proporciona, realiza las siguientes operaciones usando SKLearn:
- Elimina las filas con valores nulos y las de posiciones que no sean F, C o G (no es necesario SKLearn).
- Escala con MinMaxScaler los valores de alturas y pesos.
- Codifica con OneHotEncoder la columna "position".
- Obtén una muestra del 2% de jugadores respetando las proporciones de la columna "position".

In [269]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.model_selection import train_test_split

In [270]:
df = pd.read_csv("players.csv", sep=',')
df.head()

,id,first_name,height_feet,height_inches,last_name,position,team,weight_pounds
0,14,Ike,NaN,NaN,Anigbogu,C,"{'id': 12, 'abbreviation': 'IND', 'city': 'Ind...",NaN
1,25,Ron,NaN,NaN,Baker,G,"{'id': 20, 'abbreviation': 'NYK', 'city': 'New...",NaN
2,47,Jabari,NaN,NaN,Bird,G,"{'id': 2, 'abbreviation': 'BOS', 'city': 'Bost...",NaN
3,67,MarShon,NaN,NaN,Brooks,G,"{'id': 15, 'abbreviation': 'MEM', 'city': 'Mem...",NaN
4,71,Lorenzo,NaN,NaN,Brown,G,"{'id': 28, 'abbreviation': 'TOR', 'city': 'Tor...",NaN


### Elimina las filas con valores nulos y las de posiciones que no sean F, C o G (no es necesario SKLearn).

In [271]:
# Eliminar filas que contengan valores nulos
df = df.dropna()

In [272]:
# Convertir unidades
df['height_cm'] = df['height_feet'] * 30.48
df['weight_kg'] = df['weight_pounds'] * 0.45

In [273]:
# Eliminar columnas
df = df.drop('id', axis=1)
df = df.drop('height_feet', axis=1)
df = df.drop('height_inches', axis=1)
df = df.drop('weight_pounds', axis=1)

In [274]:
# Eliminar los jugadores cuya posición no sea F, C o G.
posiciones_validas = ['F', 'C', 'G']
df = df[df['position'].isin(posiciones_validas)]

# Resetear índice
df = df.reset_index(drop=True)

df

,first_name,last_name,position,team,height_cm,weight_kg
0,Alex,Abrines,G,"{'id': 21, 'abbreviation': 'OKC', 'city': 'Okl...",182.88,90.00
1,Kosta,Koufos,C,"{'id': 26, 'abbreviation': 'SAC', 'city': 'Sac...",213.36,110.25
2,Michael,Beasley,F,"{'id': 25, 'abbreviation': 'POR', 'city': 'Por...",182.88,105.75
3,Wade,Baldwin IV,G,"{'id': 11, 'abbreviation': 'HOU', 'city': 'Hou...",182.88,90.00
4,Jared,Terrell,G,"{'id': 18, 'abbreviation': 'MIN', 'city': 'Min...",182.88,102.15
...,...,...,...,...,...,...
376,Mitchell,Robinson,C,"{'id': 20, 'abbreviation': 'NYK', 'city': 'New...",213.36,108.00
377,Collin,Sexton,G,"{'id': 29, 'abbreviation': 'UTA', 'city': 'Uta...",182.88,85.50
378,Landry,Shamet,G,"{'id': 24, 'abbreviation': 'PHX', 'city': 'Pho...",182.88,84.60
379,Anfernee,Simons,G,"{'id': 25, 'abbreviation': 'POR', 'city': 'Por...",182.88,83.25


### Escala con MinMaxScaler los valores de alturas y pesos.

In [275]:
# Crear escalador
minmax = MinMaxScaler()

# Escalar alturas y pesos
df[['height_cm']] = minmax.fit_transform(df[['height_cm']])
df[['weight_kg']] = minmax.fit_transform(df[['weight_kg']])

df

,first_name,last_name,position,team,height_cm,weight_kg
0,Alex,Abrines,G,"{'id': 21, 'abbreviation': 'OKC', 'city': 'Okl...",0.5,0.250000
1,Kosta,Koufos,C,"{'id': 26, 'abbreviation': 'SAC', 'city': 'Sac...",1.0,0.625000
2,Michael,Beasley,F,"{'id': 25, 'abbreviation': 'POR', 'city': 'Por...",0.5,0.541667
3,Wade,Baldwin IV,G,"{'id': 11, 'abbreviation': 'HOU', 'city': 'Hou...",0.5,0.250000
4,Jared,Terrell,G,"{'id': 18, 'abbreviation': 'MIN', 'city': 'Min...",0.5,0.475000
...,...,...,...,...,...,...
376,Mitchell,Robinson,C,"{'id': 20, 'abbreviation': 'NYK', 'city': 'New...",1.0,0.583333
377,Collin,Sexton,G,"{'id': 29, 'abbreviation': 'UTA', 'city': 'Uta...",0.5,0.166667
378,Landry,Shamet,G,"{'id': 24, 'abbreviation': 'PHX', 'city': 'Pho...",0.5,0.150000
379,Anfernee,Simons,G,"{'id': 25, 'abbreviation': 'POR', 'city': 'Por...",0.5,0.125000


### Codifica con OneHotEncoder la columna "position".

In [276]:
# Crear codificador
one_hot_encoder = OneHotEncoder(sparse_output=False, dtype=int)

# Codificar posición
codigos_posicion = one_hot_encoder.fit_transform(df[['position']])
df_codigos_posicion = pd.DataFrame(
    codigos_posicion,
    columns=one_hot_encoder.get_feature_names_out(['position']),
    #index=df.index -> necesario si no reseteamos índice después de la limpieza
)

# Concatenar
df = pd.concat([df, df_codigos_posicion], axis=1)

df.head()

,first_name,last_name,position,team,height_cm,weight_kg,position_C,position_F,position_G
0,Alex,Abrines,G,"{'id': 21, 'abbreviation': 'OKC', 'city': 'Okl...",0.5,0.250000,0,0,1
1,Kosta,Koufos,C,"{'id': 26, 'abbreviation': 'SAC', 'city': 'Sac...",1.0,0.625000,1,0,0
2,Michael,Beasley,F,"{'id': 25, 'abbreviation': 'POR', 'city': 'Por...",0.5,0.541667,0,1,0
3,Wade,Baldwin IV,G,"{'id': 11, 'abbreviation': 'HOU', 'city': 'Hou...",0.5,0.250000,0,0,1
4,Jared,Terrell,G,"{'id': 18, 'abbreviation': 'MIN', 'city': 'Min...",0.5,0.475000,0,0,1


### Obtén una muestra del 2% de jugadores respetando las proporciones de la columna "position".

In [277]:
# Proporciones de la columna 'position' en el conjunto de datos original
conteo_posiciones = df['position'].value_counts()
proporcion_posiciones = conteo_posiciones / conteo_posiciones.sum()
print(proporcion_posiciones)

position
G    0.464567
F    0.406824
C    0.128609
Name: count, dtype: float64


In [278]:
df_train, df_test = train_test_split(df, test_size=0.02, stratify=df['position'], random_state=1)

print(f'DF Original: {df.shape}')
print(f'DF Train: {df_train.shape}')
print(f'DF Test: {df_test.shape}')

df_test

DF Original: (381, 9)
DF Train: (373, 9)
DF Test: (8, 9)


,first_name,last_name,position,team,height_cm,weight_kg,position_C,position_F,position_G
275,Jevon,Carter,G,"{'id': 17, 'abbreviation': 'MIL', 'city': 'Mil...",0.5,0.216667,0,0,1
308,Alex,Len,C,"{'id': 26, 'abbreviation': 'SAC', 'city': 'Sac...",1.0,0.666667,1,0,0
161,Jake,Layman,F,"{'id': 18, 'abbreviation': 'MIN', 'city': 'Min...",0.5,0.375000,0,1,0
109,Shabazz,Napier,G,"{'id': 30, 'abbreviation': 'WAS', 'city': 'Was...",0.5,0.083333,0,0,1
361,Christian,Wood,F,"{'id': 7, 'abbreviation': 'DAL', 'city': 'Dall...",0.5,0.366667,0,1,0
184,Bryn,Forbes,G,"{'id': 18, 'abbreviation': 'MIN', 'city': 'Min...",0.5,0.166667,0,0,1
329,Chris,Boucher,F,"{'id': 28, 'abbreviation': 'TOR', 'city': 'Tor...",0.5,0.250000,0,1,0
73,Denzel,Valentine,G,"{'id': 29, 'abbreviation': 'UTA', 'city': 'Uta...",0.5,0.333333,0,0,1


In [279]:
# Proporciones de la columna 'position' en el conjunto de datos de entrenamiento
conteo_posiciones = df_train['position'].value_counts()
proporcion_posiciones = conteo_posiciones / conteo_posiciones.sum()
print(proporcion_posiciones)

position
G    0.463807
F    0.407507
C    0.128686
Name: count, dtype: float64


In [280]:
# Proporciones de la columna 'position' en el conjunto de datos de prueba
conteo_posiciones = df_test['position'].value_counts()
proporcion_posiciones = conteo_posiciones / conteo_posiciones.sum()
print(proporcion_posiciones)

position
G    0.500
F    0.375
C    0.125
Name: count, dtype: float64
